وظيفة لغز عبور الجسر

تعريف المكتبة والحقائق المستخدمة

In [ ]:
from experta import*

class Person(Fact):
    #الشخص الذي يعبر الجسر
    pass

class State(Fact):
    #حالة الجسر
    pass

class Move(Fact):
    #خطوة بين حالتين
    pass

تعريف الحالة الابتدائية و DefFacts()

In [ ]:
class BridgeExpertSystem(KnowledgeEngine):
    
    @DefFacts()
    def initial_state(self):
        yield State(
            left=('me', 'lab', 'worker', 'scientist'),
            right=(),
            light='left',
            time=0,
            path=[]
        )

قواعد توليد ابناء الحالة

In [ ]:
#قاعدة عبور شخصين من اليسار لليمين
@Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path))
def move_left_to_right(self, left, right, time, path):
    from itertools import combinations

    persons_times = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}

    for p1, p2 in combinations(left, 2):
        new_left = list(left)
        new_left.remove(p1)
        new_left.remove(p2)
        new_right = list(right) + [p1, p2]

        t = max(persons_times[p1], persons_times[p2])
        total_time = time + t

        if total_time <= 17:
            self.declare(State(
                left=tuple(new_left),
                right=tuple(new_right),
                light='right',
                time=total_time,
                path=path + [f"{p1} and {p2} crossed to right in {t} min"]
            ))


قواعد التحقق من الشروط

In [ ]:
#قاعدة العودة من اليمين الى اليسار لشخص واحد, عودة المصباح من اليمين الى اليسار
@Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path))
def move_right_to_left(self, left, right, time, path):
    persons_times = {'you': 1, 'lab': 2, 'worker': 5, 'scientist': 10}

    for p in right:
        new_right = list(right)
        new_right.remove(p)
        new_left = list(left) + [p]

        t = persons_times[p]
        total_time = time + t

        if total_time <= 17:
            self.declare(State(
                left=tuple(new_left),
                right=tuple(new_right),
                light='left',
                time=total_time,
                path=path + [f"{p} returned to left in {t} min"]
            ))


قاعدة التحقق من الوصول الى الحالة الهدف

قواعد طباعة الحل

تشغيل الخبير

In [ ]:
engine = BridgeExpertSystem()
engine.reset()
engine.run()
